# EDGE-FIRE GPU 학습
런타임 → 런타임 유형 변경 → GPU를 선택하세요. 현재 패키지는 3,564장 예비 실험 데이터이며 전체 63만 장 학습이 아닙니다. 전체 데이터 확대 시에도 장소별 분리와 검증을 유지해야 합니다.
Google Drive의 MyDrive/EDGEFIRE 폴더에 edgefire_colab_dataset.zip을 올린 뒤 순서대로 실행하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요"
print(torch.cuda.get_device_name(0))

In [ ]:
%pip install -q ultralytics==8.4.216 pyyaml
# Colab에 이미 설치된 CUDA 지원 torch를 유지합니다.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.216", "pyyaml"], check=True)
from pathlib import Path
import time, zipfile, shutil, hashlib
archive = Path("/content/drive/MyDrive/EDGEFIRE/edgefire_colab_dataset.zip")
expected_bytes = 1407549284
expected_sha256 = "6fa25b325bd2b8f7373cb2abb2deca4101212ed93d8651a89cbdba6456ebe582"
deadline = time.monotonic() + 7200
while not archive.is_file() or archive.stat().st_size != expected_bytes:
    if time.monotonic() > deadline:
        raise TimeoutError("Drive upload not complete; rerun after checking upload")
    print("Waiting for Drive upload...", flush=True)
    time.sleep(30)
local = Path("/content/edgefire_colab_dataset.zip")
shutil.copy2(archive, local)
with local.open("rb") as stream:
    actual = hashlib.file_digest(stream, "sha256").hexdigest()
assert actual == expected_sha256, "Package hash mismatch; do not train"
root = Path("/content/edgefire_colab")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(local) as z:
    for member in z.infolist():
        if not (root/member.filename).resolve().is_relative_to(root.resolve()):
            raise ValueError("Unsafe ZIP member")
    z.extractall(root)
print("Package verified and extracted. Ready for GPU training.")
run_dir = Path("/content/drive/MyDrive/EDGEFIRE/runs/pilot_3564_yolo26n_gpu_autorun")
command = [sys.executable, "-u", "/content/edgefire_colab/code/train_colab.py", "--dataset", str(root/"dataset"), "--save-dir", str(run_dir), "--model", "yolo26n-seg.pt", "--epochs", "30"]
if (run_dir/"last.pt").is_file():
    command.append("--resume")
print("Starting GPU training; checkpoints persist to", run_dir, flush=True)
subprocess.run(command, check=True)
print("GPU run completed. Do not run the older training cell below for the same experiment.")


## 실행 결과와 범위
업로드 완료를 기다린 뒤 크기와 SHA256을 검증하고 GPU 학습을 시작합니다. 체크포인트는 Drive/EDGEFIRE/runs/pilot_3564_yolo26n_gpu_autorun 에 매 회차 저장됩니다.
이 패키지는 3564장 예비 데이터입니다. 전체 63만 장 학습이나 실사용 검증 완료를 의미하지 않습니다. 새 모델은 자동으로 서비스에 배포되지 않습니다.
연결 중단 후 마지막 체크포인트가 있으면 준비 셀을 재실행해 이어갈 수 있습니다. Colab GPU 할당과 실행 시간은 보장되지 않습니다.